In [2]:
from math import log2, sqrt
import numpy as np
import scipy
import matplotlib.pyplot as plt
from scipy.signal import spectrogram
from scipy.fft import fft

from torch.nn.functional import conv1d, avg_pool2d
import torch.nn as nn
from torch.fft import fft, ifft
import torch

from einops import rearrange, repeat

# Develop band plan of time vs frequency
# Tune to band
# While still on this band
    # get 200ms of samples
    # Use polyphase filter to channelize to 100 channels
        # reshape input
        # apply filters to each channel
        # reshape
        # Do ifft
    # shift into per-channel buffer
    # Do 2.4e4 pt fft per channel
    # Estimate noise floor
    # Check if peak is above threshold
    # save channel to file if true
    
# Fs = 2.4e6
# NSAMPLES = int(Fs)
# TOVERLAP = 0.2
# NOVERLAP = int(TOVERLAP*Fs)
# x = np.random.rand(NSAMPLES) + 1j*np.random.rand(NSAMPLES)

# %timeit X = fft(x, NSAMPLES)

class PolyphaseChannelizer(nn.Module):
    def __init__(self, M: torch.Tensor, h: torch.Tensor):
        """
        Takes input x of complex samples and channelizes it into M channels with a polyphase filter constructed from h
        Initialization:
            - M The number of channels. Must be a power of 2.
            - h The prototype filter. Number of taps must be integer multiple of M.
        Inputs:
            - x The input signal(s). [B] x T, where B is batches and T is time (samples). T must be an integer multiple of M.
            - prev The last M/len(h)-1 samples of the previous correlation. [B] x M/len(h)-1
        Outputs:
            - The channelized samples [B] x M x T/M
            - The last M/len(h)-1 samples of the correlation. [B] x M/len(h)-1
        """
        super().__init__()
        assert float(int(log2(M))) == log2(M), f"M ({M}) must be a power of 2"
        assert h.size(-1) % M == 0, f"Size of filter ({h.size(-1)}) must be integer multiple of number of channels ({M})"
        self.M = M
        self.filter_bank = rearrange(h, "(T M) -> M 1 T", M=M)
        self.filter_bank = self.filter_bank.flip(-1)
        self.taps = self.filter_bank.size(-1)

    def init_prev(self):
        return torch.zeros(2, self.M, self.taps-1)

    def forward(self, x, prev):
        assert prev.size(-1) == self.taps-1, f"The last dimension of prev ({prev.size(-1)}) must be taps-1 ({self.taps-1})"
        assert prev.size(-2) == self.M, f"The second to last dimension of prev ({prev.size(-2)}) must be M ({self.M})"
        assert x.dtype in [torch.complex32, torch.complex64, torch.complex128], "x must be complex"   

        # assert prev.size()[0:-1] == x.size()[0:-1], f"The optional batch and IQ dimensions must match between x {x.size()[0:-1]} and prev {prev.size()[0:-1]}"
        assert x.size(-1) % self.M == 0
        if x.ndim == 1:
            dummy_batch = True
            x = x.unsqueeze(0)
        
        # Shift input to left in freq domain by 1/2 channel. This helps the output channels all line up right
        t = torch.arange(0, x.size(-1))
        x = x*torch.exp(-1j*2*torch.pi*t/(self.M*2))

        # Reshape into decimated streams with I/Q dimension collapsed into the batch so we can apply same filter to both
        x_down = rearrange(torch.view_as_real(x.resolve_conj()), "B (T M) IQ -> (B IQ) M T", M = self.M)

        # Do full convolution
        x_filt_temp = conv1d(x_down, self.filter_bank, groups=self.M, padding=self.taps-1)

        # Separate out parts where it only partially overlapped
        # x_filt_pre = x_filt_temp[...,:self.taps-1]
        # x_filt = x_filt_temp[...,self.taps-1:-(self.taps-1)]
        x_filt_post = x_filt_temp[...,-(self.taps-1):]

        # # # Assemble the full convolution for this frame
        # x_filt = torch.cat([x_filt_pre + prev, x_filt], dim=-1)
        # x_filt = x_filt_temp

        x_filt = x_filt_temp

        # Perform ifft across channels
        x_filt = torch.view_as_complex(rearrange(x_filt, "(B IQ) M T -> B M T IQ", IQ=2).contiguous())
        print(x_filt.shape)
        x_chan = ifft(x_filt, dim=-2, norm="forward")
        y = x_chan
        
        # Rearrange output channels
        y = torch.roll(y.flip(dims=(-2,)), 1, dims=-2)
        # circ shift in frequency domain by half channel.
        t = torch.arange(0, y.size(-1))
        y = y*torch.exp(-1j*2*torch.pi*t/2)
        # Now fft(y).flatten() ~= fft(x)

        if dummy_batch:
            y = y.squeeze(0) # Remove batch if just a dummy dim

        return y, x_filt_post

class Spectrogramer(nn.Module):
    def __init__(self, n_fft: int, hop_length=None):
        super().__init__()
        self.n_fft = n_fft
        if hop_length is None:
            hop_length = n_fft // 4
        self.hop_length = hop_length
        self.overlap_frames = (self.n_fft // self.hop_length) - 1
        self.overlap = self.n_fft - self.hop_length
        self.window = torch.hann_window(n_fft, periodic=True)

    def forward(self, x: torch.Tensor, prev: torch.Tensor):
        assert prev.size(-1) == self.overlap, f"last dimension of prev must be n_fft - hop_length = {self.overlap}"
        x = torch.cat([prev, x], dim=-1)
        y = torch.stft(x, self.n_fft, self.hop_length, window=self.window, center=False)
        prev = x[..., -self.overlap:]

        return y, prev
    
    def init_prev(self):
        return torch.zeros(self.overlap)

class EnergyDetector(nn.Module):
    def __init__(self, n_bins: int, pwr_thresh_dB: float, win_kernel_size=(4,4)):
        super().__init__()
        self.n_bins = n_bins
        self.pwr_thresh_dB = pwr_thresh_dB
        self.win_kernel_size = win_kernel_size
        
    def forward(self, X_mag: torch.Tensor):
        # x is 1024x1024 spectrogram
        
        # Estimate noise floor with time avg of avg of bin avg
        psd = X_mag.square()
        noise_floor = rearrange(psd, "M T (B F) -> M T B F", B=self.n_bins).mean(dim=-1).mean(dim=-1).mean(dim=-1)
        noise_floor_dB = 10*noise_floor.log10()
        
        # win_size = 16
        psd_avg = avg_pool2d(psd, self.win_kernel_size)
        psd_dB = 10*psd_avg.log10()

        detections = psd_dB > (noise_floor_dB + self.pwr_thresh_dB).unsqueeze(-1).unsqueeze(-1)
        chan_detects = detections.any(dim=-1).any(dim=-1)

        return chan_detects, noise_floor_dB

def gen_samps(N, M, Fs):
    """Generate 1/2 channel bandwidth BPSK signals centered in M channels with ascending amplitude"""
    samps_per_sym = M*4
    x = 2 * torch.randint(0,2,(M, N//samps_per_sym)) - 1
    x = repeat(x, "M S -> M (S rep)", rep=samps_per_sym)
    t = torch.arange(0, N).unsqueeze(0)
    f = torch.arange(start=Fs/M/2, step=Fs/M, end=Fs).unsqueeze(-1)
    shifts = torch.exp(1j*2*torch.pi*f*t/Fs)
    x_shifted = x * shifts
    x_shifted = x_shifted * torch.linspace(start=0.3, end=1.0, steps = M).unsqueeze(-1)
    x = x_shifted.sum(0)
    return x

def gen_single_channel_signal(N, M, Fs, k=4):
    t = torch.arange(0, N)
    f = Fs/M/2 + k * (Fs/M)
    x = torch.exp(1j*2*torch.pi*f*t/Fs).to(dtype=torch.complex64)
    return x

# def get_samps(N, Fs):
    

In [3]:
# Setup frames
FRAMES = 16
Fs = int(2**21)
frame_s = 1
N_frame = frame_s * Fs
N = Fs*frame_s*FRAMES

# Setup channelizer
M = 128
Fs_chan = Fs // M
N_chan = frame_s*FRAMES*Fs_chan
taps = 4096
# Cutoff frequency is the edge of the channel minus ~half the transition width to ensure nothing leaks in from the adjacent band
Fc = int(Fs / M / 2) - (5.5*Fs/taps)*0.5

# Setup Spectrogram
NFFT = 1024
hop_factor = 4
hop_length = NFFT // hop_factor

print("Cutoff Frequency:", Fc)
print("Channelized sampling f: ", Fs_chan)
print("Time resolution: ", NFFT / Fs_chan)
print("Freq resolution: ", Fs_chan // NFFT)
print(f"Dimensions (px) txf:  {int(N_chan / hop_length)} x {NFFT}")
print(f"Dimensions (ph) sxf KHz:  {N_chan} x {Fs_chan * 1e-3}")

from scipy.signal import firwin

h = torch.tensor(firwin(numtaps=taps, cutoff=Fc, window='blackmanharris', fs=Fs), dtype=torch.float32)
pc = PolyphaseChannelizer(M,h)
pc_prev = pc.init_prev()


sg = Spectrogramer(NFFT, hop_length)
sg_prev = repeat(sg.init_prev(), "N -> M N", M=M)

n_fbins = 64
pwr_thresh_dB = 15
signal_width_Hz = 512
signal_len_s = 0.5
win_kernel_size = (signal_width_Hz // (Fs_chan // NFFT), int( (N_chan/hop_length) / (FRAMES*frame_s) * signal_len_s))
print(f"{win_kernel_size=}") 
ed = EnergyDetector(n_fbins, pwr_thresh_dB, win_kernel_size)

Cutoff Frequency: 6784.0
Channelized sampling f:  16384
Time resolution:  0.0625
Freq resolution:  16
Dimensions (px) txf:  1024 x 1024
Dimensions (ph) sxf KHz:  262144 x 16.384
win_kernel_size=(32, 32)


In [4]:
from rtlsdr import RtlSdr
from scipy.io import savemat
import datetime
import matplotlib.pyplot as plt
from pathlib import Path

DIR = "detects"

# configure device
def get_sdr(Fs: int):
    sdr = RtlSdr()
    sdr.sample_rate = Fs
    sdr.set_bias_tee(True)
    sdr.set_agc_mode(True)

    return sdr

def save_detects(samples: np.ndarray, spectrograms: np.ndarray, center_frequencies: np.ndarray, noise_floor: np.ndarray, sampling_rate: int, dir):
    detects = samples.shape[0]
    detect_time = datetime.datetime.now().strftime("%Y_%m_%d_%H_%M_%S_%f")
    # vmin = spectrograms.min()
    # vmax = spectrograms.max()
    for i in range(detects):
        fstem = f"{detect_time}_{center_frequencies[i]}_{sampling_rate}"
        mat_path = Path(dir, fstem + ".mat")
        spectrogram_path = Path(dir, fstem + ".png")
        
        detect_dict = {"samples": samples[i], "fc": center_frequencies[i], "fs": sampling_rate, "time": detect_time}
        savemat(mat_path, detect_dict, do_compression=True)

        vmin = noise_floor[i]
        plt.imsave(spectrogram_path, spectrograms[i], cmap='inferno', vmin=vmin)
        # plt.imsave(spectrogram_path, spectrograms[i], cmap='inferno')

# Setup SDR
TOTAL_BW = Fs * 13
min_f = int(3e6)
max_f = min_f+TOTAL_BW
Fcs = list(range(min_f+(Fs//2), max_f, Fs))
Fcs_ofst = torch.arange(0,len(Fcs)).unsqueeze(-1) * Fs
Fcs_chan = torch.arange(Fs_chan//2, Fs, Fs_chan).unsqueeze(0) + Fcs_ofst
sdr = get_sdr(Fs)
clamp_dB = 26
x = None
# while True:
try:
    for Fc_i, Fc in enumerate(Fcs[0:1]):
        sdr.fc = Fc                        # Tune to Fc
        print(f"{Fc=}")
        x_frame = []
        print("Reading samples")
        for i in range(FRAMES):
            x_frame.append(sdr.read_samples(N_frame))  # Get samples
        
        print("Processing samples")
        x = torch.tensor(np.concatenate(x_frame, axis=-1), dtype=torch.complex64)
        # x = gen_single_channel_signal(N,M,Fs)
        x_chan, pc_prev = pc(x, pc_prev)   # Channelize
        sgs, sg_prev = sg(x_chan, sg_prev) # Spectrograms
        sgs = sgs.abs()
        detects, noise_floor_dB = ed(sgs)                  # Signal detect
        
        # Kick off thread that saves spectrograms off        
        if detects.any():
            detected_chans = detects.nonzero()
            print(f"Saving detections: {len(detected_chans)}")
            detected_samples = x_chan[detects].numpy()
            detected_spectrograms = sgs[detects]
            detected_spectrograms = (20*detected_spectrograms.log10()).clamp(max=noise_floor_dB[detects,None,None]+clamp_dB).numpy()
            detected_center_frequencies = Fcs_chan[Fc_i, detected_chans].numpy()
            save_detects(detected_samples, detected_spectrograms, detected_center_frequencies, noise_floor_dB, Fs_chan, DIR)
finally:
    sdr.close()



Fc=4048576
Reading samples
Processing samples
torch.Size([1, 128, 262175])
Saving detections: 52


In [ ]:
# x = torch.rand(Fs*16, dtype=torch.complex64)
# window = torch.hann_window(Fs//4, periodic=True)
# %timeit torch.stft(x, Fs//4, Fs//8, window=window, center=False)

# xt = x[:Fs]
# sg = Spectrogramer(32768, hop_length)
# spec_prev = sg.init_prev()
# X, prev = sg(xt, spec_prev)

# plt.imshow(20*torch.log10(X.abs()), cmap='inferno')
# plt.show()

138 ms ± 13.3 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
# plt.imshow(20*torch.log10(X.abs()), cmap='inferno', interpolation='none')

In [ ]:
# sdr.close()

In [ ]:
# sdr.center_freq = 91.7e6


# samples = sdr.read_samples(N)
# sdr.close()

# x = torch.tensor(samples, dtype=torch.complex64)
# x_chan, pc_prev = pc(x, pc_prev)

# plt.plot(20*torch.log10(fft(x).abs()))
# plt.show()

# f = torch.fft.fftfreq(int(N//M), 1/(Fs/M))
# X_chan_mag = 20*torch.log10(X_chan.abs())

# for chan in X_chan_mag[30:35]:
#     print(chan.size())
#     plt.plot(f, chan)
#     plt.show()

# def proc(x, pc_prev, pc, sg, sg_prev):
#     x_chan, pc_prev = pc(x, pc_prev)
#     sgs, sg_prev = sg(x_chan, sg_prev)
#     return sgs

# with torch.no_grad():
#     x = gen_samps(N_chan,M,Fs)
#     proc(x, pc_prev, pc, sg, sg_prev)

#     %timeit proc(x, pc_prev, pc, sg, sg_prev)

# for i in range(FRAMES):
#     x = gen_samps(N,M,Fs)
#     x_chan, pc_prev = pc(x, pc_prev)
#     sgs, sg_prev = sg(x_chan, sg_prev)
#     # Estimate noise floor
#     # Get peaks
#     # Threshold
    

# X = fft(x)
# plt.plot(10*X.abs().log10())
# plt.show()



# print(x_chan.size())

# plt.plot(x_chan[4][0:100])
# plt.show()

# X_chan_mag = 10*fft(x_chan).abs().log10()
# # f = torch.fft.fftfreq(int(N//M), 1/(Fs/M))
# # X_chan_mag = 20*torch.log10(X_chan.abs())

# plt.plot(X_chan_mag.flatten())
# plt.show()

# for chan in X_chan_mag[:4]:
#     print(chan.size())
#     plt.plot(chan)
#     plt.show()

In [ ]:
# M=5
# T=30
# IQ=2
# H=3
# x = torch.arange(0,60, step=0.2)
# xr = rearrange(x, "(IQ T) -> 1 IQ T", IQ=2)
# xc = rearrange(xr, "B IQ (T M) -> (B IQ) M T", M=M)
# xf = rearrange(torch.arange(M*H).to(dtype=torch.float32), "(M T) -> M 1 T", M=M)
# xf = xf.flip(-1)
# print(xr)
# print(xf)
# print(xc)
# xcf = conv1d(xc, xf, groups=M, padding=H-1)
# print(xcf)
# print(xr.size())
# print(xc.size())
# print(xf.size())
# print(xcf.size())